In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r"C:\Users\hp\Desktop\Amazon\CSV Files\amazon_india_2018.csv")

In [3]:
df.shape

(99495, 34)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99495 entries, 0 to 99494
Data columns (total 34 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   transaction_id          99495 non-null  object 
 1   order_date              99495 non-null  object 
 2   customer_id             99495 non-null  object 
 3   product_id              99495 non-null  object 
 4   product_name            99495 non-null  object 
 5   category                99495 non-null  object 
 6   subcategory             99495 non-null  object 
 7   brand                   99495 non-null  object 
 8   original_price_inr      99495 non-null  object 
 9   discount_percent        99495 non-null  float64
 10  discounted_price_inr    99495 non-null  float64
 11  quantity                99495 non-null  int64  
 12  subtotal_inr            99495 non-null  float64
 13  delivery_charges        91542 non-null  float64
 14  final_amount_inr        99495 non-null

In [5]:
import numpy as np

df.replace("", np.nan, inplace=True)


In [6]:
dfc = df.copy()

Question 1
Your dataset contains order_date in multiple formats: 'DD/MM/YYYY', 'DD-MM-YY', 'YYYY-MM-DD', and some invalid entries like '32/13/2020'. Clean and standardize all dates to 'YYYY-MM-DD' format, handling invalid dates appropriately.


In [7]:
import pandas as pd

dfc['order_date'] = (
    dfc['order_date']
    .astype('string')
    .str.strip()
    .str.replace(r'\s+', '', regex=True)
)

dfc['order_date'] = pd.to_datetime(
    dfc['order_date'],
    dayfirst=True,
    errors='coerce'
)
dfc['order_date'] = dfc['order_date'].dt.strftime('%Y-%m-%d')


C:\Users\hp\AppData\Local\Temp\ipykernel_5644\570715499.py:10: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dfc['order_date'] = pd.to_datetime(


In [10]:
dfc['order_date'].tail(150)

99345    2018-11-13
99346    2018-12-05
99347    2018-10-06
99348    2018-10-02
99349    2018-09-17
            ...    
99490    2018-02-03
99491    2018-11-28
99492    2018-03-11
99493    2018-11-03
99494    2018-05-19
Name: order_date, Length: 150, dtype: object

Question 2
The original_price_inr column contains mixed data types: numeric values, text with '₹' symbols, comma separators ('₹1,25,000'), and some entries like 'Price on Request'. Clean this column to contain only numeric values in Indian Rupees.

In [11]:
dfc['original_price_inr'] = (
    dfc['original_price_inr']
        .astype(str)                      
        .str.replace('₹', '', regex=False) 
        .str.replace(',', '', regex=False)
        .str.replace('Rs ', '', regex=False)
        .str.strip()                
)

dfc['original_price_inr'] = pd.to_numeric(
    dfc['original_price_inr']
)


Question 3
Customer ratings appear in various formats: '5.0', '4 stars', '3/5', '2.5/5.0', and some missing values. Standardize all ratings to numeric scale 1.0-5.0, handling inconsistent formats and missing values strategically.

In [12]:
import re

def parse_rating(r):
    if pd.isna(r):
        return np.nan

    if '/' in r:
        a, b = r.split('/')
        return float(a) / float(b) * 5

    m = re.search(r'\d+\.?\d*', r)
    return float(m.group()) if m else np.nan


dfc['customer_rating'] = dfc['customer_rating'].apply(parse_rating)

Question 4
The customer_city column has inconsistent naming: 'Bangalore/Bengaluru', 'Mumbai/Bombay', 'Delhi/New Delhi', along with spelling errors and case variations. Standardize all city names and handle geographical variations.

In [17]:
dfc['customer_city'] = (
    dfc['customer_city']
    .str.lower()
    .str.strip()
)
 
city_map = {
    'bangalore': 'Bengaluru',
    'bengaluru': 'Bengaluru',
    'bangalore/bengaluru': 'Bengaluru',
    'bengalore' : 'Bengaluru',
    'Bengaluru' : 'banglore',
    
    'mumbai': 'Mumbai',
    'bombay': 'Mumbai',
    'mumbai/bombay': 'Mumbai',
    'mumba' : 'Mumbai',
    'calcutta' : 'kolkata',

    'delhi': 'Delhi',
    'new delhi': 'Delhi',
    'delhi/new delhi': 'Delhi',
    'delhi NCR' : 'Delhi',
    'delhi ncr' : 'Delhi',

    'chenai' : 'chennai',
    'madras' : 'chennai'
}

dfc['customer_city'] = dfc['customer_city'].replace(city_map)

Question 5
Boolean columns (is_prime_member, is_prime_eligible, is_festival_sale) contain mixed values: True/False, Yes/No, 1/0, Y/N, and some missing entries. Convert all boolean columns to consistent True/False format.


In [22]:
import numpy as np

bool_cols = ['is_prime_member', 'is_prime_eligible', 'is_festival_sale']

for col in bool_cols:
    dfc[col] = dfc[col].replace(['', ' ', 'NA', 'N/A', None, 'None'], np.nan)


bool_map = {
    True: True,
    'True': True,
    'true': True,
    'Yes': True,
    'yes': True,
    'Y': True,
    'y': True,
     1: True,
    
     False: False,
    'False': False,
    'false': False,
    'No': False,
    'no': False,
    'N': False,
    'n': False,
     0: False
}


for col in bool_cols:
    dfc[col] = dfc[col].map(bool_map)

Question 6
Product categories have variations: 'Electronics/Electronic/ELECTRONICS/Electronics & Accessories'. Standardize category names across the dataset and ensure consistent naming conventions.

In [21]:
dfc.columns = dfc.columns.str.strip()

category_map = {
    'electronics': 'Electronics',
    'ELECTRONICS': 'Electronics',
    'electronics & accessories': 'Electronics',
    'Electronicss': 'Electronics',
    'Electronics & Accessories': 'Electronics',
    'Electronic': 'Electronics',
    'clothing': 'Fashion'
}

dfc['category'] = dfc['category'].replace(category_map)

Question 7
The delivery_days column contains negative values, text entries like 'Same Day', '1-2 days', and some unrealistic values like 50 days. Clean this column to contain only valid numeric delivery days.


In [23]:
import numpy as np

days_map = {
    'Express': '0',
    'Same Day': '0',
    '-1': 'None',
    '1-2 days': '2'
}

dfc['delivery_days'] = dfc['delivery_days'].replace(days_map)

Question 8
Identify and handle duplicate transactions where the same customer, product, date, and amount appear multiple times. Some duplicates are genuine (bulk orders) while others are data errors. Develop a strategy to distinguish and handle both cases.


In [24]:
dup_cols = [
    "customer_id",
    "product_id",
    "order_date",
    "final_amount_inr"
]

price_cols = [
    "original_price_inr",
    "discounted_price_inr",
    "subtotal_inr",
    "final_amount_inr"
]

dfc["dup_count"] = (
    dfc.groupby(dup_cols)["transaction_id"]
      .transform("count")
)


dfc["price_identical"] = (
    dfc.groupby(dup_cols)[price_cols]
      .transform("nunique")
      .max(axis=1) == 1
)


dfc["is_high_value"] = dfc["final_amount_inr"] > 5000
dfc["is_bulk_customer"] = dfc["customer_spending_tier"].isin(["Premium"])
dfc["is_bulk_quantity"] = dfc["quantity"] > 1

dfc["is_duplicate_candidate"] = dfc["dup_count"] > 1


In [25]:
df_deduped = dfc[dfc["is_duplicate_candidate"]].drop_duplicates(subset=dup_cols, keep="first")


In [26]:
print("Rows deleted:", (df_deduped))

Rows deleted:           transaction_id  order_date         customer_id   product_id  \
14     TXN_2018_00000015  2018-01-04  CUST_2018_00018940  PROD_000402   
108    TXN_2018_00000109  2018-01-29  CUST_2018_00025678  PROD_000072   
123    TXN_2018_00000124  2018-01-06  CUST_2018_00013289  PROD_000070   
323    TXN_2018_00000324  2018-01-24  CUST_2016_00013330  PROD_000327   
498    TXN_2018_00000499  2018-01-10  CUST_2017_00012045  PROD_001692   
...                  ...         ...                 ...          ...   
98162  TXN_2018_00098163  2018-12-15  CUST_2018_00028813  PROD_000125   
98456  TXN_2018_00098457  2018-12-25  CUST_2017_00010497  PROD_001900   
98829  TXN_2018_00098830  2018-12-30  CUST_2018_00034728  PROD_000338   
98859  TXN_2018_00098860  2018-12-15  CUST_2018_00017836  PROD_000463   
98955  TXN_2018_00098956  2018-12-17  CUST_2017_00001493  PROD_001639   

                            product_name     category  subcategory    brand  \
14             Xiaomi Poco F1 

In [56]:
len(dfc)

77385

In [27]:
cols_to_drop = [
    "dup_count",
    "price_identical",
    "is_high_value",
    "is_bulk_customer",
    "is_bulk_quantity",
    "is_duplicate_candidate"
]

dfc = dfc.drop(columns=cols_to_drop)

Question 9
The dataset contains outlier prices where some products show prices 100x higher than expected due to data entry errors (decimal point issues). Identify and correct these outliers using statistical methods and domain knowledge.


In [28]:
import numpy as np

dfc["product_median_price"] = (
    dfc.groupby("product_id")["final_amount_inr"]
      .transform("median")
)

dfc["price_outlier"] = (
    dfc["final_amount_inr"] > 50 * dfc["product_median_price"]
)

dfc.loc[dfc["price_outlier"], "final_amount_inr"] /= 100
dfc.loc[dfc["price_outlier"], "discounted_price_inr"] /= 100
dfc.loc[dfc["price_outlier"], "original_price_inr"] /= 100

dfc["subtotal_inr"] = dfc["discounted_price_inr"] * dfc["quantity"]
dfc["final_amount_inr"] = dfc["subtotal_inr"] + dfc["delivery_charges"].fillna(0)

dfc["price_corrected_flag"] = dfc["price_outlier"]

dfc.drop(columns=["product_median_price"], inplace=True)

corrected_rows = dfc[dfc["price_corrected_flag"]]

print(corrected_rows)


Empty DataFrame
Columns: [transaction_id, order_date, customer_id, product_id, product_name, category, subcategory, brand, original_price_inr, discount_percent, discounted_price_inr, quantity, subtotal_inr, delivery_charges, final_amount_inr, customer_city, customer_state, customer_tier, customer_spending_tier, customer_age_group, payment_method, delivery_days, delivery_type, is_prime_member, is_festival_sale, festival_name, customer_rating, return_status, order_month, order_year, order_quarter, product_weight_kg, is_prime_eligible, product_rating, price_outlier, price_corrected_flag]
Index: []

[0 rows x 36 columns]


In [29]:
dfc = dfc.drop(
    columns=[
        "price_outlier",
        "price_corrected_flag"       
    ]
)



Question 10
Payment methods contain inconsistent naming: 'UPI/PhonePe/GooglePay', 'Credit Card/CREDIT_CARD/CC', 'Cash on Delivery/COD/C.O.D'. Standardize payment method categories and create a clean categorical hierarchy.


In [30]:
dfc["payment_method"] = (
    dfc["payment_method"]
    .str.upper()
    .str.replace(".", "", regex=False)
    .str.strip()
)

payment_map = {
    "UPI": "UPI",
    "PHONEPE": "UPI",
    "GOOGLEPAY": "UPI",
    "GPAY": "UPI",
    "PAYTM": "UPI",

    "CREDIT CARD": "Credit Card",
    "CREDIT_CARD": "Credit Card",
    "CC": "Credit Card",

    "DEBIT CARD": "Debit Card",
    "DC": "Debit Card",

    "COD": "Cash on Delivery",
    "CASH ON DELIVERY": "Cash on Delivery",

    "NET BANKING": "Net Banking"
}

dfc["payment_method"] = dfc["payment_method"].replace(payment_map)

In [36]:
dfc['original_price_inr'] = (
    dfc['original_price_inr']
    .astype(str)
    .str.replace('-', '', regex=False)
    .astype(float)
)


In [31]:
dfc["payment_method"].unique()

array(['Cash on Delivery', 'Debit Card', 'Credit Card', 'UPI',
       'Net Banking'], dtype=object)

In [32]:
import pandas as pd

columns = [
    "transaction_id","order_date","customer_id","product_id","product_name",
    "category","subcategory","brand","original_price_inr","discount_percent",
    "discounted_price_inr","quantity","subtotal_inr"
]
unique_values = {}

for col in columns:
    if col in dfc.columns:
        unique_values[col] = sorted(dfc[col].dropna().astype(str).unique())
    else:
        unique_values[col] = []

print("Unique values for each column:\n")
for col, values in unique_values.items():
    print(f"{col}: {values}\n")


Unique values for each column:

transaction_id: ['TXN_2018_00000001', 'TXN_2018_00000002', 'TXN_2018_00000003', 'TXN_2018_00000004', 'TXN_2018_00000005', 'TXN_2018_00000006', 'TXN_2018_00000007', 'TXN_2018_00000008', 'TXN_2018_00000009', 'TXN_2018_00000010', 'TXN_2018_00000011', 'TXN_2018_00000012', 'TXN_2018_00000013', 'TXN_2018_00000014', 'TXN_2018_00000015', 'TXN_2018_00000015_DUP', 'TXN_2018_00000016', 'TXN_2018_00000017', 'TXN_2018_00000018', 'TXN_2018_00000019', 'TXN_2018_00000020', 'TXN_2018_00000021', 'TXN_2018_00000022', 'TXN_2018_00000023', 'TXN_2018_00000024', 'TXN_2018_00000025', 'TXN_2018_00000026', 'TXN_2018_00000027', 'TXN_2018_00000028', 'TXN_2018_00000029', 'TXN_2018_00000030', 'TXN_2018_00000031', 'TXN_2018_00000032', 'TXN_2018_00000033', 'TXN_2018_00000034', 'TXN_2018_00000035', 'TXN_2018_00000036', 'TXN_2018_00000037', 'TXN_2018_00000038', 'TXN_2018_00000039', 'TXN_2018_00000040', 'TXN_2018_00000041', 'TXN_2018_00000042', 'TXN_2018_00000043', 'TXN_2018_00000044', 'T

In [33]:
import pandas as pd

columns = [
    "delivery_charges",
    "final_amount_inr","customer_city","customer_state","customer_tier",
    "customer_spending_tier","customer_age_group","payment_method","delivery_days",
    "delivery_type","is_prime_member","is_festival_sale",
    "festival_name"
]
unique_values = {}

for col in columns:
    if col in dfc.columns:
        unique_values[col] = sorted(dfc[col].dropna().astype(str).unique())
    else:
        unique_values[col] = []

print("Unique values for each column:\n")
for col, values in unique_values.items():
    print(f"{col}: {values}\n")


Unique values for each column:

delivery_charges: ['0.0']

final_amount_inr: ['1000.7', '100007.02', '100009.24', '10001.28', '10002.9', '100022.36', '100023.87', '100028.4', '10003.91', '100041.27', '100048.87', '10005.74', '100051.26', '100051.62', '100053.16', '10006.06', '10006.76', '100064.18', '100067.34', '10007.68', '100080.08', '100081.4', '100090.62', '1001.66', '100102.66', '100117.4', '100122.53', '10013.96', '10013.97', '100133.89', '100134.22', '100134.55', '100135.21', '100135.38', '10014.46', '10014.97', '100151.03', '100163.25', '100164.12', '100165.28', '100173.71', '10019.0', '100196.12', '100197.2', '100208.8', '100209.0', '10021.93', '100217.88', '100228.19', '100229.4', '100229.55', '100240.44', '100245.52', '100247.8', '100252.64', '100253.93', '100267.74', '10027.45', '100275.33', '100288.69', '10029.54', '100290.16', '10030.2', '10031.3', '100311.78', '100317.59999999999', '100318.88', '100333.79999999999', '100337.4', '10034.76', '100342.65', '100347.57', '100

In [30]:
import pandas as pd

columns = [    "customer_rating","return_status","order_month","order_year","order_quarter",
    "product_weight_kg","is_prime_eligible","product_rating"
]
unique_values = {}

for col in columns:
    if col in dfc.columns:
        unique_values[col] = sorted(dfc[col].dropna().astype(str).unique())
    else:
        unique_values[col] = []

print("Unique values for each column:\n")
for col, values in unique_values.items():
    print(f"{col}: {values}\n")


Unique values for each column:

customer_rating: ['3.0', '3.5', '4.0', '4.5', '5.0']

return_status: ['Cancelled', 'Delivered', 'Returned']

order_month: ['1', '10', '11', '12', '2', '3', '4', '5', '6', '7', '8', '9']

order_year: ['2017']

order_quarter: ['1', '2', '3', '4']

product_weight_kg: ['0.03', '0.04', '0.05', '0.06', '0.07', '0.08', '0.1', '0.12', '0.14', '0.15', '0.16', '0.17', '0.18', '0.19', '0.2', '0.21', '0.22', '0.23', '0.24', '0.25', '0.29', '0.3', '0.32', '0.33', '0.34', '0.35', '0.4', '0.42', '0.43', '0.45', '0.46', '0.47', '0.48', '0.49', '0.52', '0.53', '0.54', '0.55', '0.56', '0.57', '0.58', '0.59', '0.62', '0.63', '0.64', '0.65', '0.66', '0.68', '0.69', '0.72', '0.73', '0.75', '0.78', '1.2', '1.21', '1.29', '1.39', '1.4', '1.46', '1.5', '1.62', '1.74', '1.76', '1.79', '1.81', '1.98', '1.99', '2.01', '2.02', '2.04', '2.06', '2.17', '2.18', '2.26', '2.27', '2.33', '2.37', '2.42', '2.49', '2.57', '2.6', '2.66', '2.68', '2.69', '2.7', '2.75', '2.8', '21.84', '24.86'

In [ ]:
dfc

In [34]:
print(len(dfc.columns))
print(dfc.columns.tolist())


34
['transaction_id', 'order_date', 'customer_id', 'product_id', 'product_name', 'category', 'subcategory', 'brand', 'original_price_inr', 'discount_percent', 'discounted_price_inr', 'quantity', 'subtotal_inr', 'delivery_charges', 'final_amount_inr', 'customer_city', 'customer_state', 'customer_tier', 'customer_spending_tier', 'customer_age_group', 'payment_method', 'delivery_days', 'delivery_type', 'is_prime_member', 'is_festival_sale', 'festival_name', 'customer_rating', 'return_status', 'order_month', 'order_year', 'order_quarter', 'product_weight_kg', 'is_prime_eligible', 'product_rating']


In [35]:
dfc[dfc.duplicated()]

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating


In [35]:
import pandas as pd

decimal_cols = dfc.select_dtypes(include=['float', 'float64']).columns

dfc[decimal_cols] = dfc[decimal_cols].round(2)
print(decimal_cols)


Index(['original_price_inr', 'discount_percent', 'discounted_price_inr',
       'subtotal_inr', 'delivery_charges', 'final_amount_inr',
       'customer_rating', 'product_weight_kg', 'product_rating'],
      dtype='object')


In [37]:
dfc.to_csv(r"C:\Users\hp\Desktop\Amazon\CSV_Clean_Files\amazon_india_2018_clean.csv",header='infer',index=False)